# ARIMA

Owner: _(name)_

**What this notebook is for.** Build the ARIMA model, choose its settings on the validation period,
run it once on the test period, and save the predictions for part 3.

**What you do.** Run the cells in order. The settings section is where you experiment (try more
settings, keep the table). The "test once" cell is run at the very end, one time. Then write the methods
text for the report (brief, section 4: how the method works, its equations, why these settings).

**Rules everyone follows** (brief, sections 3 to 5):

- Data and split come from `src/`. Never re-download, never re-split, never shuffle.
- Learn on train, choose settings on validation, touch test once at the end.
- Each forecast uses only information available at that time (a rolling forecast).
- Keep a table of the settings you tried and their validation error.
- Time the training: the report must discuss computational cost.
- Save the test predictions with `common.save_predictions(...)`; it turns them into index points for part 3.

## How ARIMA works, in plain words (for the report's methods section)

ARIMA predicts the next value from a weighted mix of the last **p** values (the AR part, "auto-regressive")
and the last **q** forecast errors (the MA part, "moving average"), after differencing the series **d**
times to make it stationary (the I part, "integrated"). Written as ARIMA(p, d, q). Our models work on the
return, which is already stationary, so d is 0 on the return (equivalently 1 on the price).

### Setup: run this cell first (nothing to change)

It fetches the repository and the data into this Colab session, loads the shared settings, and prints the
three periods: train, validation, test. The first run takes about a minute.

In [ ]:
# ---- Setup (identical in every notebook). Just run it. ----
import sys, os, pathlib, subprocess, importlib.util

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"
EXTRA_PACKAGES = ['pmdarima']      # what this notebook needs that Colab does not have by default

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. In Colab, fetch the repository (once per session). On your own computer this step is skipped.
if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    public = subprocess.run(["git", "clone", "--quiet", REPO_URL], env=env, capture_output=True).returncode == 0
    if not public:                                   # only if the repository was made private again
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            from getpass import getpass
            token = getpass("The repository is private. Paste a GitHub token: ")
        subprocess.run(["git", "clone", "--quiet", REPO_URL.replace("https://", f"https://{token}@")], env=env, check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    os.chdir("sp500-forecasting")

# 2. Install anything missing (usually only on the first run in Colab).
missing = [p for p in EXTRA_PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("installing", ", ".join(missing), "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# 3. Find the repository folder and load the shared code (src/config.py and src/common.py).
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

# 4. The data everyone uses, and the three periods in time order.
df = common.load_prices()           # columns: Open, High, Low, Close, Volume, log_return; one row per trading day
y = common.target_series(df)        # the daily log return: what the models work on
train, val, test = common.split(y)  # dates from src/config.py
print("repository:", root)
print("data:", df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train       {train.index.min().date()} to {train.index.max().date()}  ({len(train)} days)  the model learns here")
print(f"validation  {val.index.min().date()} to {val.index.max().date()}  ({len(val)} days)  you compare settings here")
print(f"test        {test.index.min().date()} to {test.index.max().date()}  ({len(test)} days)  touched once, at the very end")
df.tail(3)

## 1. Choose p, d, q (just run, then read the table)

`auto_arima` tries many (p, d, q) combinations in the stepwise order from the lecture and keeps the one
with the lowest **AICc**, a score that rewards a good fit and penalises extra parameters. Justify the
choice with part 2's results (d), the ACF and PACF, the AICc and the validation error. The manual
candidates in the next section show the choice was examined, not just accepted.

In [ ]:
import pmdarima as pm
with common.Stopwatch() as sw_auto:                       # Stopwatch measures how long the search takes
    auto = pm.auto_arima(train.to_numpy(), seasonal=False, stepwise=True, suppress_warnings=True, information_criterion="aicc", trace=True)
print(auto.summary())
print("search time (s):", round(sw_auto.seconds, 1))

## 2. Rolling one-step forecasts: validation first, then test (just run)

The model is fitted on the training period. Each day it forecasts the next value, then is told the true
value (`update`, which keeps the parameters and extends the history) before forecasting again. Nothing
after the forecast date is ever used. The table compares the automatic choice with any manual orders you
add, by validation error in index points.

In [ ]:
def walk_forward(model, series):
    preds = []
    for t, value in series.items():
        preds.append(float(np.asarray(model.predict(n_periods=1))[0]))   # forecast tomorrow
        model.update([value])                                            # then learn what really happened
    return pd.Series(preds, index=series.index)

candidates = {"auto": auto.order}          # add manual orders to compare, e.g. "ar1": (1, 0, 0), "ma1": (0, 0, 1)
table = []
for name, order in candidates.items():
    with common.Stopwatch() as sw:
        m = pm.ARIMA(order=order, suppress_warnings=True).fit(train.to_numpy())      # arrays, not dates: statsmodels cannot extend a trading-day index
    val_pred = common.to_price(walk_forward(m, val), df)                            # back to index points
    table.append({"name": name, "order": order, "AICc": m.aicc(), "val RMSE (index points)": common.rmse(common.actual_price(val.index, df), val_pred), "fit seconds": sw.seconds})
pd.DataFrame(table).set_index("name").round(3)

## 3. The chosen order: test once, in index points, and save (run once, at the end)

In [ ]:
best_order = auto.order                     # or the manual order that won on validation
with common.Stopwatch() as sw:
    model = pm.ARIMA(order=best_order, suppress_warnings=True).fit(pd.concat([train, val]).to_numpy())   # refit on train + validation
test_pred = walk_forward(model, test)
test_price = common.to_price(test_pred, df)
print("test MAE  (index points)", round(common.mae(common.actual_price(test.index, df), test_price), 2))
print("test RMSE (index points)", round(common.rmse(common.actual_price(test.index, df), test_price), 2))
common.save_predictions("arima", df, test_pred, settings={"order": best_order, "refit": "train + validation, then update() each day"}, train_seconds=sw.seconds)

## 4. Residual check: did the model leave a pattern behind? (just run)

A residual is actual minus predicted. Good residuals look like noise: no drift over time and ACF bars
inside the shaded band.

In [ ]:
resid = (common.actual_price(test.index, df) - test_price).dropna()
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
resid.plot(ax=ax[0], title="residuals over time (index points)")
plot_acf(resid, lags=30, ax=ax[1], title="ACF of residuals (should sit inside the band)")
plt.tight_layout()

## 5. Hand the results to part 3

In [ ]:
# Last step: get the two result files into GitHub so part 3 can read them.
# In Colab this downloads them to your computer; then on github.com open the results folder,
# click "Add file" > "Upload files", drop the two files, and commit.
# On your own computer: git add results/arima_predictions.csv results/arima_meta.json, then commit and push.
if IN_COLAB:
    from google.colab import files
    for name in ["arima_predictions.csv", "arima_meta.json"]:
        files.download(str(root / "results" / name))
else:
    print("saved:", root / "results" / "arima_predictions.csv")
    print("saved:", root / "results" / "arima_meta.json")
    print("now: git add results/arima_predictions.csv results/arima_meta.json  &&  git commit -m 'arima: test predictions'  &&  git push")